In [1]:
!pip install -q ultralytics pyyaml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.0/96.0 kB 4.2 MB/s eta 0:00:00


In [2]:
import ultralytics
import torch

print("Ultralytics version:", ultralytics.__version__)
print("PyTorch version:", torch.__version__)
print("CUDA khả dụng:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Chưa có GPU, hãy bật GPU trong Settings.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics version: 8.4.173
PyTorch version: 2.10.0+cu128
CUDA khả dụng: True
GPU: Tesla T4


In [3]:
from pathlib import Path
import yaml
from ultralytics import YOLO

# 1. Đường dẫn dataset trên Kaggle
dataset_dir = Path(
    "/kaggle/input/datasets/trnilm21/railway-dataset/Dataset"
)
data_yaml = dataset_dir / "data.yaml"

print("Dataset:", dataset_dir)

# 2. Kiểm tra thư mục ảnh
train_images = dataset_dir / "train" / "images"
valid_images = dataset_dir / "valid" / "images"

if not data_yaml.exists():
    raise FileNotFoundError(f"Không tìm thấy file: {data_yaml}")

if not train_images.exists():
    raise FileNotFoundError(f"Không tìm thấy: {train_images}")

if not valid_images.exists():
    raise FileNotFoundError(f"Không tìm thấy: {valid_images}")

# 3. Cấu hình data.yaml cho Kaggle
with open(data_yaml, "r", encoding="utf-8") as f:
    data = yaml.safe_load(f)

data["train"] = str(train_images)
data["val"] = str(valid_images)
data.pop("test", None)

kaggle_yaml = Path("/kaggle/working/data_kaggle.yaml")

with open(kaggle_yaml, "w", encoding="utf-8") as f:
    yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)

# 4. Khởi tạo YOLO11s pretrained
model = YOLO("yolo11s.pt")

# 5. Train
results = model.train(
    data=str(kaggle_yaml),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    patience=20,
    project="/kaggle/working/runs",
    name="railway",
    exist_ok=True,
    plots=True,
    save=True,
    seed=42,
    close_mosaic=10
)

# 6. Đường dẫn model sau khi train
print("\nĐÃ TRAIN XONG!")
print("Model tốt nhất: /kaggle/working/runs/railway/weights/best.pt")
print("Model cuối:     /kaggle/working/runs/railway/weights/last.pt")

Dataset: /kaggle/input/datasets/trnilm21/railway-dataset/Dataset
Ultralytics 8.4.173 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data_kaggle.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/50      4.94G     0.9748     0.6744      1.016        195        640: 100% ━━━━━━━━━━━━ 115/115 5.2it/s 22.2s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 5.1it/s 3.0s0.2s
                   all        460       4282      0.902      0.847      0.905      0.681

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/50      4.94G     0.9906     0.6872      1.026        303        640: 100% ━━━━━━━━━━━━ 115/115 5.0it/s 22.9s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 4.7it/s 3.2s0.2s
                   all        460       4282      0.895      0.822      0.899      0.674

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/50      4.94G     0.9872     0.6856      1.029        214        640: 100% ━━━━━━━━━━━━ 115/115 4.8it/s 23.7s0.2s
                 Class     Images

In [ ]:
import shutil

shutil.make_archive(
    '/kaggle/working/kaggle_output',
    'zip',
    '/kaggle/working'
)